### Read Customer Data

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("csv")\
    .option("inferSchema", True)\
    .option("header", True)\
    .load("/Volumes/ansh_catalog/bronze/bronze_volume/customers/")

In [0]:
display(df)

In [0]:
df.printSchema()

Change the name to upper case

In [0]:
df = df.withColumn("name", upper(col("name")))
display(df)

Extract the domain from the email

In [0]:
df = df.withColumn("domain", split(col("email"),"@")[1])
display(df)

Aggregate the data by domain

In [0]:
display(df.groupBy("domain")
        .agg(count(col("customer_id")).alias("total_customers"))
        .sort(col("total_customers").desc())
)

Add timestamp to each row

In [0]:
df = df.withColumn("processDate", current_timestamp())
display(df)

Write data into Silver

In [0]:
if spark.catalog.tableExists("ansh_catalog.silver.customers_enriched"):
    dlt_obj = DeltaTable.forName("ansh_catalog.silver.customers_enriched")
    dlt_obj.alias("t").merge(
        df.alias("s"),
        "t.customer_id = s.customer_id")\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll()\
            .execute()
else:
    df.write.format("delta")\
        .mode("append")\
        .saveAsTable("ansh_catalog.silver.customers_enriched")
